# Dataset 2: Product Classification and Clustering

This notebook covers text-based ML classification on product listing data from PriceRunner. The task is to classify 35K+ product offers into 10 categories using their textual features.

Key challenges: short text classification, vocabulary variation across merchants, clustering evaluation.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, adjusted_rand_score,
                             normalized_mutual_info_score, silhouette_score)
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded.')

## Step 2 — Load and Explore the Dataset

In [ ]:
df = pd.read_csv('product_classification.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nCategory Distribution:')
print(df['Category'].value_counts())
df.head()

In [ ]:
# Check text lengths
df['Title_Length'] = df['Product_Title'].apply(lambda x: len(str(x).split()))
print('Product Title Word Count Stats:')
print(df['Title_Length'].describe())

plt.figure(figsize=(10, 5))
df['Title_Length'].hist(bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.title('Product Title Length Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Price distribution by category
plt.figure(figsize=(12, 6))
df.boxplot(column='Price', by='Category', rot=45, figsize=(14, 6))
plt.title('Price Distribution by Category', fontsize=14)
plt.suptitle('')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

## Step 3 — Text Preprocessing

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Clean_Title'] = df['Product_Title'].apply(clean_text)
df['Clean_Description'] = df['Description'].apply(clean_text)

# Combine title and description for richer features
df['Combined_Text'] = df['Clean_Title'] + ' ' + df['Clean_Description']

print('Sample cleaned text:')
for i in range(3):
    print(f'  {df["Clean_Title"].iloc[i]}')

print(f'\nTotal records: {len(df)}')

## Step 4 — Feature Extraction (TF-IDF)

In [ ]:
# Encode target
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])
print('Categories:', dict(zip(le.classes_, range(len(le.classes_)))))

# TF-IDF on combined text
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_tfidf = tfidf.fit_transform(df['Combined_Text'])
y = df['Category_Encoded']

print(f'\nTF-IDF Matrix Shape: {X_tfidf.shape}')
print(f'Vocabulary Size: {len(tfidf.vocabulary_)}')

In [ ]:
# Top terms per category
feature_names = tfidf.get_feature_names_out()
for cat_idx in range(len(le.classes_)):
    cat_mask = (y == cat_idx)
    cat_tfidf_mean = X_tfidf[cat_mask].mean(axis=0).A1
    top_indices = cat_tfidf_mean.argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    print(f'{le.classes_[cat_idx]:>20s}: {" | ".join(top_terms)}')

## Step 5 — Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## Step 6 — Classification Models

In [ ]:
# Model 1: Multinomial Naive Bayes (baseline for text)
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print('MODEL 1 — Multinomial Naive Bayes')
print('=' * 45)
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_nb, average="macro"):.4f}')

In [ ]:
# Model 2: Linear SVM
svm = LinearSVC(max_iter=2000, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print('MODEL 2 — Linear SVM')
print('=' * 35)
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_svm, average="macro"):.4f}')

In [ ]:
# Model 3: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('MODEL 3 — Logistic Regression')
print('=' * 40)
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_lr, average="macro"):.4f}')

In [ ]:
# Model comparison summary
models_results = {
    'Naive Bayes': f1_score(y_test, y_pred_nb, average='macro'),
    'Linear SVM': f1_score(y_test, y_pred_svm, average='macro'),
    'Logistic Regression': f1_score(y_test, y_pred_lr, average='macro')
}

plt.figure(figsize=(8, 5))
plt.bar(models_results.keys(), models_results.values(), color=['#3498db','#e74c3c','#2ecc71'])
plt.ylabel('Macro F1-Score')
plt.title('Classification Model Comparison', fontsize=14)
plt.ylim(0, 1)
for i, (name, score) in enumerate(models_results.items()):
    plt.text(i, score + 0.02, f'{score:.3f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## Step 7 — Confusion Matrix (Best Model)

In [ ]:
# Find best model
best_name = max(models_results, key=models_results.get)
print(f'Best Model: {best_name}\n')

# Use SVM predictions for confusion matrix (usually best for text)
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Linear SVM', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 8 — Clustering (K-Means)

In [ ]:
# K-Means clustering with k=10 (matching number of categories)
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans.fit_predict(X_tfidf)

# Evaluate clustering against true labels
ari = adjusted_rand_score(y, cluster_labels)
nmi = normalized_mutual_info_score(y, cluster_labels)

print('CLUSTERING RESULTS — K-Means (k=10)')
print('=' * 45)
print(f'Adjusted Rand Index (ARI): {ari:.4f}')
print(f'Normalized Mutual Information (NMI): {nmi:.4f}')
print(f'\nCluster Size Distribution:')
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# Visualize clusters using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray()[:5000])  # Subset for speed

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y[:5000], cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, label='Category')
plt.title('PCA Projection — True Categories (first 5K samples)', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

## Step 9 — Cross-Validation (Final Comparison)

In [ ]:
# Stratified 5-fold CV for all models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in [('Naive Bayes', MultinomialNB(alpha=0.1)),
                     ('Linear SVM', LinearSVC(max_iter=2000, random_state=42)),
                     ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42))]:
    scores = cross_val_score(model, X_tfidf, y, cv=cv, scoring='f1_macro')
    cv_results[name] = scores
    print(f'{name}: CV Macro F1 = {scores.mean():.4f} (+/- {scores.std():.4f})')

plt.figure(figsize=(8, 5))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.ylabel('Macro F1-Score')
plt.title('5-Fold CV Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Step 10 — Summary and Key Takeaways

### Results Summary

- TF-IDF with bigrams captures enough signal from short product titles
- Linear SVM typically outperforms Naive Bayes on text classification
- K-Means clustering shows partial alignment with true categories
- PCA visualization reveals some category separation in the feature space

### What This Dataset Taught You

1. Short text classification requires careful feature engineering
2. TF-IDF with n-grams is a strong baseline before moving to embeddings
3. Linear models (SVM, LR) work surprisingly well on high-dimensional sparse text features
4. Clustering quality metrics (ARI, NMI) are essential — visual inspection alone is not enough
5. Product titles carry more classification signal than product descriptions